# Day 15: Introduction to Agentic AI

Walkthrough of agents, tools, reasoning, error handling, memory, multi-step planning, evaluation, real-world use cases, and agents combined with RAG.

## 1. What are agents

An agent thinks, decides, and acts in a loop, instead of a single retrieve-then-answer pass like RAG.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
print("Agent loop: THINK -> DECIDE -> ACT -> OBSERVE -> repeat")
print("RAG answers questions. Agents use RAG (and other tools) to get things done.")

## 2. Simple agent example

The smallest possible agent loop, using a scripted fake LLM so it runs without an API key.

In [ ]:
from simple_agent import Agent, TOOLS

agent = Agent(tools=TOOLS)
history = agent.run("what is the capital of France? then email the answer to ops@example.com")
print(f"\nTotal tool calls: {len(history)}")

## 3. Tool registry

A structured way to register tools with names, descriptions, and parameters -- auto-generating the documentation an LLM would see.

In [ ]:
from tool_registry import ToolRegistry, example_search, example_calculate

registry = ToolRegistry()
registry.register("search", example_search, "Searches for information matching a query.", ["query"])
registry.register("calculate", example_calculate, "Evaluates a simple arithmetic expression.", ["expression"])

print(registry.get_documentation())

## 4. Reasoning steps

Breaking the agent's decision into explicit think / decide / act / observe / reflect steps.

In [ ]:
from agent_reasoning import think, decide, act, observe, reflect, TOOLS as REASONING_TOOLS

context = []
thought = think("How many millions of people live in France?", context)
decision = decide(thought, context)
result = act(decision, REASONING_TOOLS)
observation = observe(decision, result)
context.append(observation)

print(f"THINK:   {thought}")
print(f"DECIDE:  {decision}")
print(f"OBSERVE: {observation}")
print(f"REFLECT: {reflect(context)}")

## 5. Error handling

Tool failures, unexpected results, infinite loops, and fallbacks -- all recoverable instead of crashing the agent.

In [ ]:
from error_handling import call_tool_safely, flaky_divide

print(call_tool_safely(flaky_divide, 10, 0))
print(call_tool_safely(flaky_divide, 10, 2))

## 6. Memory

Short-term memory for the current task, long-term memory persisted to disk across runs.

In [ ]:
import os
from agent_memory import ShortTermMemory, LongTermMemory, agent_answer_question, MEMORY_FILE

if os.path.exists(MEMORY_FILE):
    os.remove(MEMORY_FILE)

short_term = ShortTermMemory()
long_term = LongTermMemory()
log = []

answer1, path1 = agent_answer_question("What is the meaning of life?", short_term, long_term, log)
short_term.clear()
answer2, path2 = agent_answer_question("What is the meaning of life?", short_term, long_term, log)

print(f"First call path:  {path1}")
print(f"Second call path: {path2}")
print(f"Real tool calls made: {len(log)}")

os.remove(MEMORY_FILE)

## 7. Multi-step tasks

Decomposing a goal into a plan, executing it, and recovering from a transient step failure.

In [ ]:
from multi_step_tasks import plan_task, execute_plan

plan = plan_task("onboard new customer")
print(f"Plan has {len(plan)} steps\n")
results = execute_plan(plan)
completed = sum(1 for r in results if r["ok"])
print(f"\nCompleted {completed}/{len(plan)} steps")

## 8. Agent evaluation

Scoring agent runs on completion rate, efficiency, and tool error rate instead of just eyeballing the output.

In [ ]:
from agent_evaluation import TEST_SET, run_agent_on_task, evaluate

results = [run_agent_on_task(t) for t in TEST_SET]
report = evaluate(results)

print(f"Completion rate: {report['completion_rate']:.0%}")
print(f"Average steps: {report['average_steps']:.1f}")
print(f"Average tool errors: {report['tool_error_rate']:.1f}")

## 9. Real-world examples

Four realistic agents built from the same shared tools: support, research, shopping, and code.

In [ ]:
from real_world_examples import support_agent, shopping_agent

support_agent("I have a question about RAG.")
print()
shopping_agent(budget=900)

## 10. Agent plus RAG

Wiring the agent to the real Day 5/6/7 retrieval + ranking + generation pipeline, and comparing it against a guess-only tool with no access to real documents.

In [ ]:
sys.path.insert(0, os.path.abspath('../day-05-retrieval'))
sys.path.insert(0, os.path.abspath('../day-06-ranking'))
sys.path.insert(0, os.path.abspath('../day-07-generation'))

from integration_with_previous_days import RagTool, guess_tool, demonstrate_rag_vs_no_rag

rag_tool = RagTool()
demonstrate_rag_vs_no_rag("What is RAG?", rag_tool)

## 11. Key takeaways

- An agent is RAG plus the ability to DO things -- think, decide, act, observe, repeat.
- Tools should be small, well-documented, and predictable: a usable result or a labeled error, never an uncaught crash.
- Explicit reasoning steps (think/decide/act/observe/reflect) make an agent's behavior debuggable instead of a black box.
- Error handling is load-bearing for agents, not optional: max-step guards, safe tool wrapping, and fallbacks all matter because agents run unattended.
- Short-term memory serves the current task; long-term memory, persisted to disk, is what lets an agent get faster or smarter across separate runs.
- Complex goals need to be decomposed into a plan, and the agent needs to tolerate (and retry past) transient step failures.
- Agent quality is measurable: completion rate, steps taken, and tool error rate turn "seems to work" into real numbers.
- RAG gives an agent grounded answers instead of guesses; the agent then decides what to DO with that grounded answer -- the one thing RAG alone never provided.